# ECE 506 FFN — PYNQ-Z2

Audited local package; real-board testing pending. Start with B11. Use your assigned PYNQ-Z2 and run one cell at a time. Do not use Run All. Read README.md before programming. Stop on any error. A Python timeout cannot recover a blocked AXI/MMIO access.


## 1. Read-only preflight
No PYNQ import or hardware access. Run this from the extracted package directory. Use a fresh kernel; after a previous session, restart only following a successful `session.close()`.


In [ ]:
from pathlib import Path
import sys
package = Path.cwd().resolve()
sys.path.insert(0, str(package))
from ffn_host import FFNSession
session = FFNSession(package)
print(session.manifest['config'])
print('PREFLIGHT PASS; physical board not accessed')


## 2. Program and check clock
The cooperative lock remains held until a normal close. This cell does not read FFN registers. Wait for `programmed` at 100 MHz before proceeding.


In [ ]:
session.program()


## 3. First PL access and unloaded-state checks
If this cell hangs, stop and arrange board recovery; do not rerun download or restart the kernel. Success returns `guards: pass`.


In [ ]:
session.check_control()


## 4. Allocate and clean buffers
No inference has been launched. Verify stage `allocated`.


In [ ]:
session.allocate_buffers()


## 5. One known input
Success requires all 64 outputs to match.


In [ ]:
session.smoke()


## 6. Complete correctness suite
Nine parameter sets, 206 inputs, including signed routing, ReLU, rounding, saturation, persistence and replacement.


In [ ]:
validation = session.validate()
print('Validated inferences:', sum(r['inferences'] for r in validation))
validation


## 7. Measure FPGA host latency
20 warm-ups and 3 × 1000 calls. Input copy/flush, control, polling and output invalidate/copy are included; comparison and file I/O are excluded.


In [ ]:
session.benchmark()


## 8. Normal cleanup
Run only after successful operation. It deliberately refuses to free buffers after uncertain MMIO, timeout or mismatch.


In [ ]:
session.close()
print('CLOSED: buffers released; another package may now be tested')


## 9. Board CPU reference
This compiles the included C++ source on ARM, validates both layers and records CPU samples. Do not run it on the build machine as a board result.


In [ ]:
import subprocess
subprocess.run([sys.executable, 'run_cpu.py'], cwd=str(package), check=True)


## 10. Save evidence
Download these files with the package manifest. Repeat for K44 only after B11 has passed and closed normally. CPU compute time and Python/FPGA end-to-end time have different boundaries; report both.


In [ ]:
for name in ['manifest.json', 'load_stages.jsonl', 'board_validation.json', 'board_raw.csv', 'board_results.json', 'cpu_run.log', 'cpu_raw.csv', 'cpu_results.json']:
    p = package / name
    print(name, p.stat().st_size if p.exists() else 'MISSING')
